In [3]:
import os
import pandas as pd
from Bio import SeqIO

# =========================
# PATHS
# =========================

slow_dir = "../results/signalp_fast_slow"

species_dirs = sorted([
    d for d in os.listdir(slow_dir)
    if os.path.isdir(os.path.join(slow_dir, d))
])

print(f"Found {len(species_dirs)} species")

Found 29 species


In [1]:


# =========================
# MAIN LOOP
# =========================

for species in species_dirs:

    print(f"\nProcessing: {species}")

    species_dir = os.path.join(
        slow_dir,
        species
    )

    prediction_file = os.path.join(
        species_dir,
        "prediction_results.txt"
    )

    input_fasta = os.path.join(
        species_dir,
        "fast_secretome_full_length.fasta"
    )

    output_fasta = os.path.join(
        species_dir,
        "validated_signalp_predictions.fasta"
    )

    # -------------------------
    # Load slow predictions
    # -------------------------

    df = pd.read_csv(
        prediction_file,
        sep="\t",
        skiprows=1
    )

    df.columns = [
        "protein_id",
        "Prediction",
        "OTHER",
        "SP",
        "CS_Position"
    ]

    # -------------------------
    # Extract validated SP ids
    # -------------------------

    sp_ids = set()

    for pid in df[df["Prediction"] == "SP"]["protein_id"]:

        try:
            uid = pid.split("|")[1]
            sp_ids.add(uid)

        except:
            continue

    print(f"Validated SP proteins: {len(sp_ids)}")

    # -------------------------
    # Filter FASTA
    # -------------------------

    records = []

    for record in SeqIO.parse(input_fasta, "fasta"):

        try:
            uid = record.id.split("|")[1]

        except:
            continue

        if uid in sp_ids:
            records.append(record)

    print(f"Recovered sequences: {len(records)}")

    # -------------------------
    # Save validated secretome
    # -------------------------

    SeqIO.write(
        records,
        output_fasta,
        "fasta"
    )

print("\nDone.")

Found 29 species

Processing: aspergillus_brasiliensis
Validated SP proteins: 961
Recovered sequences: 961

Processing: aspergillus_clavatus
Validated SP proteins: 651
Recovered sequences: 651

Processing: aspergillus_fumigatus
Validated SP proteins: 718
Recovered sequences: 718

Processing: aspergillus_japonicus
Validated SP proteins: 923
Recovered sequences: 923

Processing: aspergillus_nidulans
Validated SP proteins: 787
Recovered sequences: 787

Processing: aspergillus_niger
Validated SP proteins: 913
Recovered sequences: 913

Processing: aspergillus_oryzae
Validated SP proteins: 763
Recovered sequences: 763

Processing: aspergillus_terreus
Validated SP proteins: 880
Recovered sequences: 880

Processing: chaetomium_globosum
Validated SP proteins: 974
Recovered sequences: 974

Processing: chaetomium_thermophilum
Validated SP proteins: 512
Recovered sequences: 512

Processing: fusarium_graminearum
Validated SP proteins: 1259
Recovered sequences: 1259

Processing: fusarium_oxysporum
V

In [4]:

# =========================
# BUILD COMBINED FASTA
# =========================

combined_fasta = os.path.join(
    slow_dir,
    "all_signalp_predictions.fasta"
)

all_records = []
protein_species = []

for species in species_dirs:

    fasta_file = os.path.join(
        slow_dir,
        species,
        "validated_signalp_predictions.fasta"
    )

    if not os.path.exists(fasta_file):
        continue

    for record in SeqIO.parse(fasta_file, "fasta"):

        all_records.append(record)

        protein_species.append({
            "protein": record.id,
            "species": species
        })

SeqIO.write(
    all_records,
    combined_fasta,
    "fasta"
)

pd.DataFrame(
    protein_species
).to_csv(
    os.path.join(
        slow_dir,
        "protein_species_map.csv"
    ),
    index=False
)

print(
    f"\nCombined FASTA saved:"
    f"\n{combined_fasta}"
)

print(
    f"Total proteins: {len(all_records)}"
)


Combined FASTA saved:
../results/signalp_fast_slow\all_signalp_predictions.fasta
Total proteins: 23417


In [5]:
signalp_summary = pd.DataFrame(
    protein_species
)

signalp_summary.groupby("species").size().to_csv(
    os.path.join(
        slow_dir,
        "signalp_species_counts.csv"
    )
)